# Day 4-01｜準備影片、執行球模型與 ByteTrack 追蹤
> Python 籃球運動資料分析課程  
> 這一節我會帶你們先把自己的投籃影片放到正確位置，再直接使用課程提供的球模型做推論與追蹤。  
> 我們不需要訓練模型，重點是先確認模型真的看得到球，並分清楚 detector 和 tracker 各自負責什麼。

## 你們會完成什麼
- 把自己的影片存進 Google Drive，或直接從 Colab 的瀏覽器上傳。
- 把不同格式的原始影片統一轉成 `assets/converted/student_video.mp4`。
- 抽 5 個 frame 檢查單幀球偵測，再產生 ByteTrack 預覽影片。
- 讀懂球框、confidence 與 track ID，並把問題定位在偵測或追蹤階段。

## 怎樣算完成
- 5 張檢查圖中的球框確實落在籃球上，不是只看 confidence 高就算正確。
- 預覽影片的框大致跟著球移動；短暫漏框可以發生，但 track ID 改變代表追蹤曾中斷。
- 你們能說出：單一 frame 就看不到球是 detector 問題；看得到但跨 frame 接不起來才是 tracker 問題。

## 本節產出
- `assets/results/d4_01_test_infer_grid.png`
- `assets/results/d4_01_bytetrack_ball_preview.mp4`
- `assets/results/d4_01_bytetrack_ball_preview.json`


## Setup｜先確認 Colab 與 Google Drive

如果你們使用 Google Colab，請先執行 `init_colab.ipynb`，再從 Google Drive 的
`basketball_hackathon/course/day4/` 開啟本 notebook。我建議把執行階段切成 **T4 GPU**；
Apple Silicon Mac 則會自動使用 **MPS（Metal GPU）**。

下一個程式格只負責掛載、尋找課程根目錄和檢查三個模型，所以我不會逐行拆解。執行後請確認：

- `results directory` 是 `/content/drive/MyDrive/basketball_hackathon/course/assets/results`。
- 三個模型路徑都位於 `assets/models/`，而且沒有出現檔案不完整的錯誤。

這樣後面產生的圖片、JSON 和影片才會真的留在你們的 Google Drive。


In [ ]:
from pathlib import Path
import subprocess
import sys

DRIVE_MOUNTED = False
if "google.colab" in sys.modules:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("目前這個 Colab runtime 不支援 Drive 掛載，改用 /content 本機路徑。")

COURSE_ROOT_HINT = (
    Path("/content/drive/MyDrive/basketball_hackathon/course")
    if DRIVE_MOUNTED
    else next(
        (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (p / "src" / "course_setup.py").exists()),
        Path("/content/basketball_hackathon/course"),
    )
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo, prepare_day4_workspace  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT, mount_drive=False)
DAY4_MODELS, RESULTS = prepare_day4_workspace(COURSE_ROOT)
print("Day 4 models:", {name: str(path) for name, path in DAY4_MODELS.items()})
print("results directory:", RESULTS)


## Step 1｜把自己的影片放到固定位置

我建議你們優先使用 **方式 A**，因為 Colab 重新連線後檔案仍在 Drive：

1. 在 Google Drive 進入 `我的雲端硬碟/basketball_hackathon/course/assets/raw/`。
2. 把影片拖進資料夾，例如 `my_shot.mov`。
3. 在下面把 `VIDEO_FILENAME` 改成 `"my_shot.mov"`，再執行程式格。

如果影片還在電腦，可以使用 **方式 B**：把 `USE_BROWSER_UPLOAD` 改成 `True`。執行後選取影片，
程式會先把檔案寫入 Drive 的 `assets/raw/`，再轉成 `assets/converted/student_video.mp4`。

兩個開關都維持預設值時，我會使用課程內建的 `assets/converted/video_001.mp4`。執行後請記住
`using video` 的完整路徑；Day 4-02 和 Day 4-03 要分析同一支影片。


In [ ]:
from src.video_utils import (
    convert_video,
    display_video_in_notebook,
    list_videos,
    pick_first_converted_video,
    save_uploaded_videos,
)

# ===== 你們只需要先改這兩個設定 =====
USE_BROWSER_UPLOAD = False  # 方式 B：要從電腦選檔時才改成 True
VIDEO_FILENAME = ""         # 方式 A 範例："my_shot.mov"；留空就用課程影片

raw_video_dir = COURSE_ROOT / "assets" / "raw"
converted_video_dir = COURSE_ROOT / "assets" / "converted"
raw_video_dir.mkdir(parents=True, exist_ok=True)
converted_video_dir.mkdir(parents=True, exist_ok=True)

selected_source = None
if USE_BROWSER_UPLOAD:
    # files.upload() 只負責打開選檔視窗；save_uploaded_videos 會把內容寫進 Drive。
    if "google.colab" not in sys.modules:
        raise RuntimeError("瀏覽器上傳只在 Google Colab 使用；本機請把檔案放進 assets/raw/。")
    from google.colab import files

    uploaded_files = files.upload()
    saved_uploads = save_uploaded_videos(uploaded_files, raw_video_dir)
    selected_source = saved_uploads[0]
elif VIDEO_FILENAME.strip():
    # Path(...).name 只保留檔名，避免不小心把檔案寫到課程資料夾以外。
    selected_source = raw_video_dir / Path(VIDEO_FILENAME).name
    if not selected_source.exists():
        raise FileNotFoundError(
            f"找不到 {selected_source}。請確認檔名和 Drive 的 assets/raw/ 內容完全一致。"
        )

if selected_source is not None:
    # 統一成 30 FPS、最長邊 1280 的 MP4，讓後面三份 notebook 使用同一個輸入。
    video_path = convert_video(
        selected_source,
        converted_video_dir / "student_video.mp4",
        fps=30,
        max_side=1280,
    )
else:
    video_path = pick_first_converted_video(COURSE_ROOT)

print("raw folder:", raw_video_dir)
print("converted folder:", converted_video_dir)
print("using video:", video_path)
display_video_in_notebook(video_path, width=720, muted=True, loop=True)


## Step 2｜載入球模型並設定推論參數

這個程式格會完成四件事：匯入影像工具、讀取剛才選好的影片、載入 `ball_rimV8.pt`，以及設定
後面推論會共用的參數。

- `SANITY_CONF`：單幀檢查的最低 confidence。
- `PREVIEW_CONF`：動態預覽的最低 confidence。
- `PREVIEW_IMGSZ`：送進模型的影像尺寸；球很小時較大的尺寸通常保留較多細節，但速度較慢。
- `HOLD_LAST_BALL_FRAMES`：球短暫漏偵時，預覽最多保留前一個位置幾個 frame。

這個模型同時知道球和籃框類別，但本節只保留球。執行後請確認 `model class names` 和
`keep class names`，不要只看到程式成功就直接往下跑。


In [ ]:
import cv2
import numpy as np

from src.cv_utils import save_image_rgb, show_image, side_by_side
from src.yolo_utils import (
    draw_detection_records,
    load_yolo_model,
    preferred_inference_device,
    read_video_frame,
    run_detector_on_image,
    write_bytetrack_preview_video,
)

converted = list_videos(COURSE_ROOT / "assets" / "converted")
model_path = DAY4_MODELS["ball_rim"]
DEVICE = preferred_inference_device()

# 先用相同門檻檢查單幀與影片，方便比較 detector 和 tracker 的差異。
SANITY_CONF = 0.25
PREVIEW_CONF = 0.25
PREVIEW_IMGSZ = 1280
HOLD_LAST_BALL_FRAMES = 4

# Ultralytics YOLO 會從 assets/models/ 直接載入課程權重。
model = load_yolo_model(model_path)
model_names = getattr(model, "names", {}) or {}
class_names_override = (
    [str(model_names[index]) for index in sorted(model_names)]
    if isinstance(model_names, dict)
    else list(model_names)
)
keep_class_names = [
    name for name in class_names_override
    if name.lower() in {"ball", "basketball", "item"}
]
if not keep_class_names:
    keep_class_names = class_names_override or ["ball"]

# OpenCV 只在這裡讀影片基本資料，還沒有開始逐 frame 推論。
capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise FileNotFoundError(video_path)
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
capture.release()

print("converted videos:", [path.name for path in converted])
print("using video:", video_path)
print("model path:", model_path)
print("inference device:", DEVICE)
print("model class names:", class_names_override)
print("keep class names:", keep_class_names)
print("sanity conf:", SANITY_CONF)
print("preview conf:", PREVIEW_CONF)
print("preview imgsz:", PREVIEW_IMGSZ)
print("hold last ball frames:", HOLD_LAST_BALL_FRAMES)
print("total frames:", total_frames)


## Step 3｜抽 5 個 frame，先檢查 detector

這一步只問「單張畫面裡有沒有找到球」，還不使用 ByteTrack。我會從前 180 frames 等距抽 5 張，
每張只保留 confidence 最高的球框，讓你們先專心確認位置。

請依序看：畫面有球時是否畫框、框中心是否真的落在球上、沒有球時是否產生錯誤球框。高速移動、
球太小或被手遮住都可能漏偵；如果 5 張幾乎都抓不到，先記錄 frame 編號，不要急著調 tracker。


In [ ]:
if total_frames <= 0:
    raise RuntimeError("無法取得影片總 frame 數。")

# 在分析範圍內等距抽樣，避免只看到影片開頭的相似畫面。
sample_stop = max(0, min(total_frames - 1, 180))
frame_indices = sorted({int(value) for value in np.linspace(0, sample_stop, num=5)})
while len(frame_indices) < 5 and frame_indices[-1] < total_frames - 1:
    frame_indices.append(frame_indices[-1] + 1)
frame_indices = frame_indices[:5]

rendered_tiles = []
summary_rows = []
for frame_index in frame_indices:
    image_rgb = read_video_frame(video_path, frame_index=frame_index)
    detections, _ = run_detector_on_image(
        model_path,
        image_rgb,
        conf=SANITY_CONF,
        imgsz=PREVIEW_IMGSZ,
        frame_index=frame_index,
        class_names_override=class_names_override,
        device=DEVICE,
    )
    # 只保留球類別，再取最高 confidence 的一顆作為課堂快速檢查。
    detections = [detection for detection in detections if detection.class_name in keep_class_names]
    detections = sorted(detections, key=lambda detection: detection.confidence, reverse=True)[:1]
    visualized_rgb = draw_detection_records(image_rgb, detections)
    rendered_tiles.append(visualized_rgb)
    summary_rows.append({
        "frame": frame_index,
        "detections": len(detections),
        "classes": [detection.class_name for detection in detections],
        "confidences": [round(detection.confidence, 3) for detection in detections],
    })
    print(summary_rows[-1])

# 把 5 張結果排成一張檢查圖，方便在 Drive 下載或放進報告。
top_row = side_by_side(rendered_tiles[0], rendered_tiles[1], max_width=2200)
top_row = side_by_side(top_row, rendered_tiles[2], max_width=2200)
bottom_row = side_by_side(rendered_tiles[3], rendered_tiles[4], max_width=2200)
grid_rgb = side_by_side(top_row, bottom_row, max_width=2200)

grid_path = RESULTS / "d4_01_test_infer_grid.png"
save_image_rgb(grid_path, grid_rgb)
print("saved grid:", grid_path)
show_image(grid_rgb, title="Day 4-01 ball detector sanity check", figsize=(18, 12))


## Step 4｜把 detector 接到 ByteTrack

確認單幀結果後，我們才檢查跨 frame 的連續性。YOLO 每一幀重新找球；ByteTrack 則根據框的位置、
移動和前後關係，嘗試替同一顆球維持相同 `track_id`。

播放結果時請同時看框是否跟著球、同一顆球的 ID 是否連續，以及漏框後能否重新接回。這份預覽每個
frame 最多保留一顆球，目的是教你們區分偵測與追蹤；Day 4-03 會重新保留所有候選球，再依投籃手腕
與事件連續性選出真正的投球軌跡。


In [ ]:
preview_mp4 = RESULTS / "d4_01_bytetrack_ball_preview.mp4"

# write_bytetrack_preview_video 會逐 frame 執行 YOLO，再把 detections 交給 ByteTrack。
preview_mp4, preview_records = write_bytetrack_preview_video(
    video_path=video_path,
    model_path=model_path,
    output_path=preview_mp4,
    max_frames=180,
    conf=PREVIEW_CONF,
    imgsz=PREVIEW_IMGSZ,
    class_names_override=class_names_override,
    keep_class_names=keep_class_names,
    max_detections_per_frame=1,
    hold_last_ball_frames=HOLD_LAST_BALL_FRAMES,
    device=DEVICE,
)
preview_json = preview_mp4.with_suffix(".json")

print("saved preview:", preview_mp4)
print("saved json:", preview_json)
print("records:", len(preview_records))
display_video_in_notebook(preview_mp4, width=780, muted=True, loop=True)


## Checks｜交給下一節前，我會請你們確認

- `using video` 是你們要分析的影片，而且已經位於 `assets/converted/`。
- 檢查圖裡至少有數個正確球框，並記下漏偵或誤偵的 frame。
- 預覽影片裡的框大致跟得上球，且你們能辨認 track ID 中斷。
- 不把 top-1 預覽 JSON 當成最終投球軌跡；Day 4-03 會針對每次投籃重新挑球。

所有結果都在 `assets/results/`。確認完成後，我們到 Day 4-02 找出投籃者骨架、投籃候選時段與
逐 frame 關節角度。
